In [1]:
## Load prerequisites libraries
import numpy as np
import pandas as pd
import json
#import pyarrow as pa #remove if not required 
#import pyarrow.parquet as pq #remove if not required 
import glob 
import os

import config

In [2]:
for customer in config.customerList:
    with open('credentials.json') as data_file:
        credentials = json.load(data_file)
    # Check lims dsn connection to database before proceeding
    def get_lims_connection():
        global customerName
        customerName = customer
        if config.dbType == "oracle":
            import oracledb
            oracledb.init_oracle_client(config.lib_dir)
            odbcLimsConnection = oracledb.connect(user=credentials[customerName].get("limsdb_username"), 
                                                password=credentials[customerName].get("limsdb_password"),
                                                dsn=credentials[customerName].get("limsdb_dsn"))
            odbcLimsConnection.arraysize = 5000
            
        elif config.dbType == "postgres":
            import psycopg2
            odbcLimsConnection = psycopg2.connect('dbname=' + credentials[customerName].get("limsdb_database") + ' user=' + credentials[customerName].get("limsdb_username") + 
                                                ' password=' + credentials[customerName].get("limsdb_password") + ' host=' + credentials[customerName].get("limsdb_hostname"))
            odbcLimsConnection.autocommit = True
            return odbcLimsConnection

        else:
            import pyodbc
            odbcLimsConnection = pyodbc.connect('DRIVER=' + credentials.limsdb_driver + ';DATABASE=' + credentials[customerName].get("limsdb_database") +
                                ';UID=' + credentials[customerName].get("limsdb_username") + ';PWD=' + credentials[customerName].get("limsdb_password") +
                                ';SERVER=' + credentials[customerName].get("limsdb_hostname") + ';PORT=' + credentials[customerName].get("limsdb_port") +
                                ';Trusted_Connection=yes')
                   
    
        if __name__ == '__main__':
            try:  # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
                get_lims_connection()
            except Exception as error:
                exit("Connection could not be made due to the following error: \n", error)

    finalInference = pd.DataFrame()
    for auditLog in config.auditList:
        #finalInference.iloc[0:0] 
        try:
            baseInferencePath = f'{config.data_location}Inference/IF_{auditLog}_{customer}_base_finalDF.parquet'
            if os.path.exists(baseInferencePath):
                baseInference = pd.read_parquet(baseInferencePath,engine='pyarrow')
            else:
                print(f'Error Base Inference file {baseInferencePath} not present. No inference made.')
                continue
        
            filterTypeInference = pd.DataFrame()
            FilterTypeFiles = glob.glob(f'{config.data_location}Inference/IF_{auditLog}_*_{customer}_audit_type_finalDF.parquet')
            
            for filepath in FilterTypeFiles:
                try:
                    tempInference = pd.read_parquet(filepath, engine='pyarrow')
                    filterTypeInference = pd.concat([filterTypeInference, tempInference], ignore_index=True, axis=0)  
                except Exception as e:
                    (f'Error reading {filepath}: {e}.')
                    continue

            auditLogInference = pd.concat([baseInference,filterTypeInference], ignore_index=True, axis=0) 
            finalInference = pd.concat([finalInference,auditLogInference], ignore_index=True, axis=0)

        except Exception as e:
            print(f'Error processing {auditLog}: {e}')
            continue

        # ## Query current log count
        cursor = get_lims_connection().cursor()
        sql_command = f"select MAX(COUNTER) from {config.outputTable}"
        cursor.execute(sql_command)
        response = cursor.fetchone()
        counter = response[0]

        ## Depending on the value, default either to 1 if empty or add 1 for next insert sequence
        if counter is None:
            id_number = 1
        else:
            id_number = counter + 1

        finalInference.drop_duplicates()
        counterData = (id_number) + np.arange(finalInference.shape[0])
        
        finalInference['counter'] = counterData
        finalInference = finalInference.rename(columns=str.lower)
        finalInference.dropna(how='any',axis=0,inplace=True)

        if auditLog == 'Sample':
            #print(type(finalInference)) ##troubleshooting
            sampleNumberArr = finalInference['sample_number'].unique()
            for sampleNumber in sampleNumberArr:
                cursor.execute("update sample set t0083_anomaly = 'T' where sample_number = %s",(sampleNumber,))
            for index, row in finalInference.iterrows():
                #if config.dbType == "postgres":
                cursor.execute("insert into t0083_ar(counter,t0083_type,audit_table_name,audit_table_key,audit_action," 
                            "audit_counter,audit_trans_order,audit_type,audit_timestamp,user_name,audit_main_id," 
                            "detection_method,anomaly,score,shap_local,shap_global,t0083_timestamp,reviewed) VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", 
                            (row['counter'],row['t0083_type'],row['table_name'],row['table_key'],row['audit_action'],
                            row['audit_counter'],row['trans_order'],row['audit_type'],row['audit_timestamp'],row['user_name'],row['sample_number'],
                            row['detection_method'],row['anomaly'],row['anomaly_score'],row['shap_local'],row['shap_global'],row['t0083_timestamp'],row['reviewed']))
                #else:
                #    print(f'{config.dbType} table insert is not supported. Please configure')
        elif auditLog == 'Experiment':
            print(type(finalInference)) ##troubleshooting
            for index, row in finalInference.iterrows():
                #if config.dbType == "postgres":
                cursor.execute("insert into t0083_ar(counter,t0083_type,audit_table_name,audit_table_key,audit_action," 
                            "audit_counter,audit_trans_order,audit_type,audit_timestamp,user_name,audit_main_id,audit_workbook_number,audit_document_number,audit_col_num,audit_row_num," 
                            "detection_method,anomaly,score,shap_local,shap_global,t0083_timestamp,reviewed) VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", 
                            (row['counter'],row['t0083_type'],row['table_name'],row['table_key'],row['audit_action'],
                            row['audit_counter'],row['trans_order'],row['audit_type'],row['audit_timestamp'],row['user_name'],row['experiment_number'],row['workbook_number'],row['document_number'],row['col_num'],row['row_num'],
                            row['detection_method'],row['anomaly'],row['anomaly_score'],row['shap_local'],row['shap_global'],row['t0083_timestamp'],row['reviewed']))
        elif auditLog == 'Table':
            ##troubleshooting
            for index, row in finalInference.iterrows():
                #if config.dbType == "postgres":
                cursor.execute("insert into t0083_ar(counter,t0083_type,audit_table_name,audit_parent_table,audit_action," 
                            "audit_counter,audit_trans_order,audit_type,audit_timestamp,user_name,audit_main_id," 
                            "detection_method,anomaly,score,shap_local,shap_global,t0083_timestamp,reviewed) VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", 
                            (row['counter'],row['t0083_type'],row['table_name'],row['parent_table'],row['audit_action'],
                            row['audit_counter'],row['trans_order'],row['audit_type'],row['audit_timestamp'],row['user_name'],row['parent_key'],
                            row['detection_method'],row['anomaly'],row['anomaly_score'],row['shap_local'],row['shap_global'],row['t0083_timestamp'],row['reviewed']))
        else: 
            print(f'{auditLog} not completed.')        
        
get_lims_connection().close()
cursor.close()